# FlowGraph | Transaction review with hourly graph snapshots

Which transfers should an analyst review first when the review budget is 100?

On synthetic AMLworld HI-Small, graph-aware XGBoost found **98 labeled cases in 100 alerts**. Transaction-only XGBoost found **54** on the same test transfers.

Run the cells in order. The default run checks the code and displays saved benchmark results. Set `RUN_FULL_BENCHMARK = True` to train and score all models.

This project is a prototype and is not financial advice.

## 1. Set the environment

Create a Python 3.9 environment. Install the packages below in that environment. Select its Jupyter kernel.

```sh
python -m pip install jupyterlab numpy==2.0.2 polars==1.32.3 torch==2.8.0 torch-geometric==2.6.1 scikit-learn==1.6.1 xgboost==2.1.4
```

On macOS, install the OpenMP runtime for XGBoost. For a full run, download `HI-Small_Trans.csv` and set `CSV_PATH`. The full run uses CPU memory.

In [1]:
import copy
import csv
import hashlib
import json
import platform
from dataclasses import dataclass
from importlib.metadata import version
from pathlib import Path
from tempfile import TemporaryDirectory
from time import perf_counter

import numpy as np
import polars as pl
import torch
from sklearn.metrics import auc, average_precision_score, precision_recall_curve
from torch import nn
from torch_geometric.nn import SAGEConv

CSV_PATH = Path("HI-Small_Trans.csv")
RUN_DIRECTORY = Path("flowgraph_notebook_run")
RUN_FULL_BENCHMARK = False
EPOCHS = 6
BATCH_SIZE = 384

HOUR = 3_600_000_000
FANOUTS = (4, 2)
SEED = 7
BUDGET = 100

## 2. Read the transfers

Use Polars to read the CSV. Keep bank and account identifiers as strings. Assign the source row number before sorting. Keep separate records for transfers with the same values.

In [2]:
IBM_HEADER = (
    "Timestamp", "From Bank", "Account", "To Bank", "Account",
    "Amount Received", "Receiving Currency", "Amount Paid",
    "Payment Currency", "Payment Format", "Is Laundering",
)
COLUMNS = (
    "timestamp", "from_bank", "from_account", "to_bank", "to_account",
    "amount_received", "receiving_currency", "amount_paid",
    "payment_currency", "payment_format", "is_laundering",
)
CATEGORICAL = ("receiving_currency", "payment_currency", "payment_format")
AMOUNTS = ("amount_received", "amount_paid")
TIMESTAMP_FORMAT = "%Y/%m/%d %H:%M"


def read_ibm_csv(path):
    """Assign a zero-based source record ID before filtering or sorting."""
    path = Path(path)
    with path.open(encoding="utf-8-sig", newline="") as source:
        if tuple(next(csv.reader(source), ())) != IBM_HEADER:
            raise ValueError("Expected the original IBM HI-Small transaction header.")
    with path.open("rb") as source:
        digest = hashlib.sha256()
        for chunk in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(chunk)
    frame = pl.read_csv(path, new_columns=COLUMNS, infer_schema=False).with_row_index(
        "transaction_id"
    )
    frame = frame.with_columns(
        pl.col("timestamp").str.strptime(pl.Datetime("us"), TIMESTAMP_FORMAT),
        pl.col(*AMOUNTS).cast(pl.Float64),
        pl.col("is_laundering").cast(pl.UInt8),
        pl.concat_str(["from_bank", "from_account"], separator=":").alias("sender_id"),
        pl.concat_str(["to_bank", "to_account"], separator=":").alias("receiver_id"),
    )
    return frame, digest.hexdigest()


def chronological_splits(times, cutoffs=None):
    """Return positional splits, keeping each timestamp in one partition."""
    if len(times) < 3 or np.any(times[1:] < times[:-1]):
        raise ValueError("Expected at least three chronologically sorted rows.")
    if cutoffs is None:
        cutoffs = times[len(times) * np.array([70, 85]) // 100]
    train_end, val_end = np.searchsorted(times, cutoffs, side="left")
    if not 0 < train_end < val_end < len(times):
        raise ValueError("Cutoffs must yield three nonempty timestamp-disjoint splits.")
    return {
        "train": slice(0, train_end),
        "validation": slice(train_end, val_end),
        "test": slice(val_end, len(times)),
    }


@dataclass
class History:
    offsets: np.ndarray
    times: np.ndarray
    neighbors: np.ndarray
    rows: np.ndarray

    @classmethod
    def build(cls, center, neighbor, times, account_count):
        order = np.argsort(center, kind="stable")
        counts = np.bincount(center, minlength=account_count)
        offsets = np.r_[0, counts.cumsum()]
        return cls(offsets, times[order], neighbor[order], order.astype(np.int32))

    def before(self, accounts, cutoffs):
        """Vectorized lower bound inside each account's sorted adjacency list."""
        start = self.offsets[accounts]
        low, high = start.copy(), self.offsets[accounts + 1].copy()
        while np.any(low < high):
            active = low < high
            middle = (low + high) // 2
            earlier = self.times[np.minimum(middle, len(self.times) - 1)] < cutoffs
            low = np.where(active & earlier, middle + 1, low)
            high = np.where(active & ~earlier, middle, high)
        return start, low

    def recent(self, accounts, cutoffs, fanout):
        start, stop = self.before(accounts, cutoffs)
        positions = stop[:, None] - 1 - np.arange(fanout)
        target, rank = np.nonzero(positions >= start[:, None])
        return target, positions[target, rank]


@dataclass
class Transactions:
    ids: np.ndarray
    times: np.ndarray
    sender: np.ndarray
    receiver: np.ndarray
    labels: np.ndarray
    amounts: np.ndarray
    categories: np.ndarray
    vocabularies: dict
    splits: dict
    outgoing: History
    incoming: History
    source_sha256: str
    historical: np.ndarray = None
    history_profile: str = None

    def edge_features(self, rows):
        parts = [self.amounts[rows]]
        for column, name in enumerate(CATEGORICAL):
            codes = self.categories[rows, column]
            one_hot = np.zeros((len(rows), len(self.vocabularies[name]) + 1), np.float32)
            one_hot[np.arange(len(rows)), codes] = 1
            parts.append(one_hot)
        if self.historical is not None:
            parts.append(self.historical[rows])
        return np.concatenate(parts, axis=1)

    def node_features(self, keys):
        accounts, cutoffs = keys.T
        parts = []
        for history in (self.outgoing, self.incoming):
            start, stop = history.before(accounts, cutoffs)
            count = stop - start
            age_hours = np.zeros(len(keys), dtype=np.float32)
            seen = count > 0
            age_hours[seen] = (
                cutoffs[seen] - history.times[stop[seen] - 1]
            ) / 3_600_000_000
            parts.extend((np.log1p(count), np.log1p(age_hours)))
        return np.column_stack(parts).astype(np.float32)


def prepare(frame, source_sha256, cutoffs=None):
    """Use raw IBM record IDs, timestamp[us], and bank:account endpoint strings."""
    fields = (
        "transaction_id", "timestamp", "sender_id", "receiver_id",
        "is_laundering", *AMOUNTS, *CATEGORICAL,
    )
    frame = frame.select(fields)
    if frame.schema["timestamp"] != pl.Datetime("us"):
        raise ValueError("timestamp must be timezone-naive Polars Datetime('us').")
    if any(frame.null_count().row(0)):
        raise ValueError("Missing values in required transaction columns.")
    ids = frame["transaction_id"]
    if not ids.dtype.is_integer() or ids.n_unique() != len(frame) or not len(frame) or ids.min() < 0:
        raise ValueError("transaction_id must contain unique nonnegative source record IDs.")
    frame = frame.sort(["timestamp", "transaction_id"])
    ids = frame["transaction_id"].cast(pl.Int64).to_numpy()
    times = frame["timestamp"].cast(pl.Int64).to_numpy()
    splits = chronological_splits(times, cutoffs)
    labels = frame["is_laundering"].to_numpy()
    if not np.isin(labels, [0, 1]).all():
        raise ValueError("Expected binary laundering labels.")
    labels = labels.astype(np.uint8)
    amounts = frame.select(AMOUNTS).to_numpy().astype(np.float32)
    if not np.isfinite(amounts).all():
        raise ValueError("Expected finite amounts.")
    amounts = np.sign(amounts) * np.log1p(np.abs(amounts))

    endpoints = pl.concat([frame["sender_id"], frame["receiver_id"]]).cast(pl.Categorical)
    sender, receiver = np.split(endpoints.to_physical().cast(pl.Int32).to_numpy(), 2)
    account_count = int(max(sender.max(), receiver.max())) + 1
    vocabularies, codes = {}, []
    for name in CATEGORICAL:
        values = frame[name]
        vocab = values[:splits["train"].stop].unique().sort().to_list()
        vocabularies[name] = vocab
        mapping = {value: index + 1 for index, value in enumerate(vocab)}
        codes.append(values.replace_strict(mapping, default=0, return_dtype=pl.Int32).to_numpy())

    return Transactions(
        ids, times, sender, receiver, labels, amounts, np.column_stack(codes),
        vocabularies, splits,
        History.build(sender, receiver, times, account_count),
        History.build(receiver, sender, times, account_count),
        source_sha256,
    )

## 3. Build the hourly snapshots

Use the start of the target hour as the cutoff. Use only transfers before that cutoff for both GraphSAGE stages. Use the same cutoff for the account features.

Keep incoming and outgoing transfers separate. Calculate amounts by currency. Use the target transfer's own fields in its score.

In [3]:
def unique_keys(keys):
    accounts = keys[:, 0]
    times = keys[:, 1]
    base = int(times.min())
    span = int(times.max()) - base
    if span >= 2**32:
        times_unique, time_codes = np.unique(times, return_inverse=True)
        packed = (accounts << 32) | time_codes
        unique, inverse = np.unique(packed, return_inverse=True)
        return np.column_stack((unique >> 32, times_unique[unique & 0xffffffff])), inverse
    packed = (accounts << 32) | (times - base)
    unique, inverse = np.unique(packed, return_inverse=True)
    return np.column_stack((unique >> 32, (unique & 0xffffffff) + base)), inverse



class SnapshotFeatures:
    def __init__(self, data):
        self.data = data
        self.cutoffs = data.times // HOUR * HOUR
        self.accounts = len(data.incoming.offsets) - 1
        self.currency_history = []
        for history, currency, amount in ((data.outgoing, 1, 1), (data.incoming, 0, 0)):
            vocab = len(data.vocabularies[("receiving_currency", "payment_currency")[currency]])
            codes = data.categories[history.rows, currency]
            raw = data.amounts[history.rows, amount].astype(np.float64)
            raw = np.sign(raw) * np.expm1(np.abs(raw))
            before = np.repeat(np.arange(self.accounts, dtype=np.int32), np.diff(history.offsets))
            order = np.lexsort((history.times, before.astype(np.int64) * (vocab + 1) + codes))
            groups = before[order].astype(np.int64) * (vocab + 1) + codes[order]
            boundaries = np.searchsorted(groups, np.arange(self.accounts * (vocab + 1) + 1))
            self.currency_history.append((
                vocab, boundaries, history.times[order],
                np.r_[0., np.cumsum(raw[order])], np.r_[0., np.cumsum(raw[order] ** 2)],
            ))

    @property
    def node_dim(self):
        return 12 + sum(4 * item[0] for item in self.currency_history)

    def nodes(self, keys):
        account, cutoff = keys.T
        features = []
        for history, item in zip((self.data.outgoing, self.data.incoming), self.currency_history):
            start, stop = history.before(account, cutoff)
            _, recent = history.before(account, cutoff - HOUR)
            _, reference = history.before(account, cutoff - 7 * HOUR)
            count = stop - start
            seen = count > 0
            last = history.times[np.maximum(stop - 1, 0)]
            first = history.times[np.minimum(start, len(history.times) - 1)]
            features.extend((
                np.log1p(count), seen, np.log1p(np.where(seen, (cutoff-last)/HOUR, 0)),
                np.log1p(np.where(seen, (cutoff-first)/HOUR, 0)),
                np.log1p(stop-recent), np.log1p((recent-reference)/6),
            ))
            vocab, bounds, times, sums, squares = item
            groups = account[:, None] * (vocab + 1) + np.arange(1, vocab + 1)
            lo = bounds[groups]
            left, right = lo.copy(), bounds[groups+1].copy()
            while np.any(left < right):
                active = left < right
                middle = (left + right) // 2
                earlier = times[np.minimum(middle, len(times)-1)] < cutoff[:, None]
                left = np.where(active & earlier, middle+1, left)
                right = np.where(active & ~earlier, middle, right)
            n = left-lo
            total = np.where(n, sums[left]-sums[lo], 0)
            mean = total / np.maximum(n, 1)
            variance = np.maximum((squares[left]-squares[lo])/np.maximum(n, 1)-mean**2, 0)
            features.extend((np.log1p(n), np.sign(total)*np.log1p(abs(total)),
                             np.sign(mean)*np.log1p(abs(mean)), np.log1p(np.sqrt(variance))))
        return np.concatenate([np.reshape(x, (len(keys), -1)) for x in features], axis=1).astype(np.float32) * .2

    def roots(self, rows):
        data = self.data
        accounts = np.column_stack((data.sender[rows], data.receiver[rows])).reshape(-1)
        return unique_keys(np.column_stack((accounts, np.repeat(self.cutoffs[rows], 2))))

    def block(self, targets, fanout):
        sources = [targets]
        destinations = []
        evidence = []
        for history in (self.data.incoming, self.data.outgoing):
            dest, pos = history.recent(targets[:, 0], targets[:, 1], fanout)
            sources.append(np.column_stack((history.neighbors[pos], targets[dest, 1])))
            destinations.append(dest)
            evidence.append(history.rows[pos])
        keys, inverse = unique_keys(np.concatenate(sources))
        root = inverse[:len(targets)]
        split = len(targets) + len(destinations[0])
        edges = (np.stack((inverse[len(targets):split], destinations[0])),
                 np.stack((inverse[split:], destinations[1])))
        return keys, (root, edges, evidence)

## 4. Define the models and measures

GraphSAGE uses two stages. Each stage has separate incoming and outgoing means. The classifier uses the sender and receiver embeddings and the target fields.

Train the no-neighbor model separately. Keep its endpoint features and classifier inputs. Use only observed transfers for training.

AP means average precision. Precision@100 is the fraction of labeled cases in the first 100 results. Sort equal scores by transaction ID.

In [4]:
def batches(rows, size):
    for start in range(0, len(rows), size):
        yield rows[start:start + size]



def metrics(ids, labels, scores):
    if not np.isfinite(scores).all():
        raise ValueError("Expected finite ranking scores.")
    ranked = labels[np.lexsort((ids, -scores))]
    positives = int(labels.sum())
    if positives:
        precision, recall, _ = precision_recall_curve(labels, scores)
    return {
        "rows": len(labels), "positives": positives, "prevalence": float(labels.mean()),
        "average_precision": float(average_precision_score(labels, scores)) if positives else None,
        "pr_auc_trapezoid": float(auc(recall, precision)) if positives else None,
        **{f"precision_at_{k}": float(ranked[:k].mean()) if len(ranked) >= k else None
           for k in (50, 100, 500, 1000)},
    }

In [5]:
class DirectedSAGE(nn.Module):
    def __init__(self, node_dim, edge_dim, neighbors=True):
        super().__init__()
        self.neighbors = neighbors
        self.root = nn.ModuleList([nn.Linear(node_dim, 64), nn.Linear(64, 64)])
        self.incoming = nn.ModuleList([SAGEConv((d,d), 64, root_weight=False, bias=False)
                                      for d in (node_dim,64)])
        self.outgoing = nn.ModuleList([SAGEConv((d,d), 64, root_weight=False, bias=False)
                                      for d in (node_dim,64)])
        self.head = nn.Sequential(nn.Linear(128+edge_dim,64), nn.ReLU(),
                                  nn.Dropout(.1), nn.Linear(64,1))

    def score(self, features, rows):
        device = self.head[0].weight.device
        def tensor(values):
            return torch.as_tensor(values, device=device)
        roots, endpoints = features.roots(rows)
        attrs = tensor(features.data.edge_features(rows))
        if self.neighbors:
            middle, inner = features.block(roots, FANOUTS[0])
            outer_keys, outer = features.block(middle, FANOUTS[1])
            x = tensor(features.nodes(outer_keys))
            for layer, (target, edges, _) in enumerate((outer,inner)):
                root = x[tensor(target)]
                x = (self.root[layer](root) + self.incoming[layer]((x,root),tensor(edges[0]))
                     + self.outgoing[layer]((x,root),tensor(edges[1]))).relu()
        else:
            x = tensor(features.nodes(roots))
            for layer in self.root:
                x = layer(x).relu()
        return self.head(torch.cat((x[tensor(endpoints)].reshape(len(rows),-1),attrs),1)).flatten()


@torch.inference_mode()
def predict(model, features, rows, batch_size):
    model.eval()
    return np.concatenate([model.score(features, chunk).cpu().numpy()
                           for chunk in batches(rows, batch_size)])


def fit_sage(features, train, validation, *, neighbors, epochs, batch_size):
    labels = features.data.labels
    positive, negative = train[labels[train]==1], train[labels[train]==0]
    rng = np.random.default_rng(SEED)
    torch.manual_seed(SEED)
    model = DirectedSAGE(features.node_dim, features.data.edge_features(train[:1]).shape[1],
                         neighbors)
    optimizer = torch.optim.AdamW(model.parameters(), lr=.0015)
    best, state, chosen = -1, None, None
    for epoch in range(1,epochs+1):
        model.train()
        for pos in batches(rng.permutation(positive), max(1,batch_size//64)):
            neg = rng.choice(negative, min(len(negative),batch_size-len(pos)), replace=False)
            scores = model.score(features, np.r_[pos,neg])
            loss = .5*(nn.functional.softplus(-scores[:len(pos)]).mean()
                       + nn.functional.softplus(scores[len(pos):]).mean())
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
        val = predict(model, features, validation, batch_size)
        ap = average_precision_score(labels[validation],val)
        print(f"sage neighbors={neighbors} epoch={epoch} validation AP={ap:.6f}",flush=True)
        if ap > best:
            best, state, chosen = ap, copy.deepcopy(model.state_dict()), epoch
    model.load_state_dict(state)
    return model, chosen


def rule_scores(data, train, test):
    scores = np.zeros(len(test),dtype=np.float32)
    for currency in np.unique(data.categories[train,1]):
        sorted_amounts = np.sort(data.amounts[train[data.categories[train,1]==currency],1])
        mask = data.categories[test,1]==currency
        scores[mask] = np.searchsorted(sorted_amounts,data.amounts[test[mask],1],side="right") / len(sorted_amounts)
    return scores

## 5. Check the cutoff

Run the checks on the small synthetic fixture. Check both sampling stages for same-hour transfers. Check that future transfers and changes to holdout labels do not change the score.

In [6]:
from datetime import datetime, timedelta

def write_fixture(path):
    start = datetime(2022, 9, 1)
    rows = []
    for i in range(720):
        time = start + timedelta(minutes=i // 3)
        rows.append([time.strftime(TIMESTAMP_FORMAT), "001", f"{i % 13:04x}",
                     "002", f"{i % 17:04x}", i + 1, "US Dollar", i + 1,
                     "US Dollar", "ACH", int(i % 17 == 0)])
    rows[1] = rows[0]
    with path.open("w", newline="") as output:
        writer = csv.writer(output)
        writer.writerow(IBM_HEADER)
        writer.writerows(reversed(rows))

In [7]:
def check_cutoffs(fixture_path):
    raw, fp = read_ibm_csv(fixture_path)
    bounds = (int(raw["timestamp"].cast(pl.Int64).min())+HOUR,
              int(raw["timestamp"].cast(pl.Int64).min())+2*HOUR)
    base=prepare(raw,fp,bounds)
    f=SnapshotFeatures(base)
    i=np.flatnonzero(base.times > bounds[0])[:8]
    roots,_=f.roots(i)
    middle, inner=f.block(roots,4)
    _,outer=f.block(middle,2)
    for _,_,evidence in (inner,outer):
        for rows in evidence:
            assert np.all(base.times[rows]<bounds[0])
    model=DirectedSAGE(f.node_dim,base.edge_features(i).shape[1]).eval()
    with torch.no_grad():
        expected=model.score(f,i).numpy()
    future=raw.tail(1).with_columns(pl.lit(len(raw)).alias("transaction_id").cast(raw.schema["transaction_id"]),
         pl.lit(bounds[1]+HOUR).cast(pl.Datetime("us")).alias("timestamp"))
    mutated=pl.concat((raw.reverse(),future)).with_columns(
        pl.when(pl.col("timestamp").cast(pl.Int64)>=bounds[0])
        .then(1-pl.col("is_laundering")).otherwise(pl.col("is_laundering")).alias("is_laundering"))
    other=prepare(mutated,fp,bounds)
    other_features = SnapshotFeatures(other)
    other_roots, _ = other_features.roots(i)
    np.testing.assert_allclose(f.nodes(roots), other_features.nodes(other_roots), atol=1e-6)
    with torch.no_grad():
        actual=model.score(other_features,i).numpy()
    np.testing.assert_allclose(expected,actual,atol=1e-6)
    assert len(np.unique(other.ids))==len(raw)+1
    assert metrics(np.array([2,1]),np.array([1,0]),np.array([1.,1.]))["average_precision"]==.5
    print("Same-bucket and future exclusion at both hops, tied input-order invariance, future append and holdout-label invariance passed.")

with TemporaryDirectory() as directory:
    fixture_path = Path(directory) / "fixture.csv"
    write_fixture(fixture_path)
    check_cutoffs(fixture_path)

Same-bucket and future exclusion at both hops, tied input-order invariance, future append and holdout-label invariance passed.


## 6. Run the benchmark

Split the data in time order. Move split boundaries to the start of an hour. Use validation AP to select the GraphSAGE checkpoints.

Score the same test transfers with each model. Keep labels in a separate evaluation file. Use a new output directory for each run.

In [8]:
def run(csv, directory, epochs=6, batch_size=384, scope_rows=None):
    from xgboost import XGBClassifier

    if epochs < 1 or batch_size < 2 or (scope_rows is not None and scope_rows < 3):
        raise ValueError("Expected positive epochs, batch size ≥2, and at least three scope rows.")

    started = perf_counter()
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=False)
    raw, fingerprint = read_ibm_csv(csv)
    source_rows = len(raw)
    sorted_times = raw["timestamp"].cast(pl.Int64).sort().to_numpy()
    scope_end = None
    if scope_rows is not None and scope_rows < source_rows:
        scope_end = int(sorted_times[scope_rows] // HOUR * HOUR)
        raw = raw.filter(pl.col("timestamp").cast(pl.Int64) < scope_end)
        sorted_times = sorted_times[:len(raw)]
    bounds = sorted_times[np.array([7*len(raw)//10,85*len(raw)//100])] // HOUR * HOUR
    data = prepare(raw, fingerprint, bounds)
    del raw
    features = SnapshotFeatures(data)
    train, validation, test = (np.arange(s.start,s.stop) for s in data.splits.values())
    torch.set_num_threads(4)
    labels, results, choices = data.labels, {}, {}
    if any(np.unique(labels[rows]).size < 2 for rows in (train,validation,test)):
        raise ValueError("Every partition must contain both classes for this experiment.")
    run_id = f"{fingerprint[:12]}-hourly-v1-{len(data.ids)}-e{epochs}"
    pl.DataFrame({"transaction_id":data.ids[test],"label":labels[test]}).write_parquet(
        directory/"test_labels.parquet")
    root = np.column_stack((data.sender[test],data.receiver[test]))
    unseen = np.zeros(len(test),dtype=np.uint8)
    for accounts in root.T:
        unseen += sum(h.before(accounts, features.cutoffs[test])[1]-h.offsets[accounts]
                      for h in (data.incoming,data.outgoing)) == 0

    def save(name, scores):
        ids = data.ids[test]
        report = metrics(ids,labels[test],scores)
        ranked = np.lexsort((ids,-scores))[:BUDGET]
        tp = int(labels[test][ranked].sum())
        positives = int(labels[test].sum())
        report["recall_at_100"] = tp/positives if positives else None
        report["confusion_at_100"] = dict(tp=tp,fp=len(ranked)-tp,fn=positives-tp,
                                        tn=len(test)-len(ranked)-positives+tp)
        report["coverage"] = 1.0
        report["history_subgroups"] = {
            str(i): metrics(ids[unseen==i],labels[test][unseen==i],scores[unseen==i])
            for i in range(3) if np.any(unseen==i)
        }
        days = (features.cutoffs[test]-bounds[1])//(24*HOUR)
        report["daily"] = {str(i): metrics(ids[days==i],labels[test][days==i],scores[days==i])
                           for i in np.unique(days)}
        results[name] = report
        pl.DataFrame({"transaction_id":ids,"target_time":data.times[test],
                      "context_cutoff":features.cutoffs[test],"score":scores,
                      "snapshot_id":features.cutoffs[test],"unseen_endpoints":unseen,
                      "model_id":[name]*len(test),"status":["available"]*len(test),
                      "run_id":[run_id]*len(test),
                      "rank_scope":["test"]*len(test)}).write_parquet(directory/f"{name}.parquet")
        print(name,report["average_precision"],report["precision_at_100"],flush=True)

    save("currency_rule",rule_scores(data,train,test))
    for historical in (False,True):
        name = "graph_xgboost" if historical else "transaction_xgboost"
        def matrix(rows):
            target = data.edge_features(rows)
            if not historical:
                return target
            roots, endpoints = features.roots(rows)
            return np.column_stack((target,features.nodes(roots)[endpoints].reshape(len(rows),-1)))
        width = matrix(train[:1]).shape[1]
        inputs = np.empty((len(train),width),dtype=np.float32)
        for chunk in batches(train,32_768):
            inputs[chunk] = matrix(chunk)
        model = XGBClassifier(n_estimators=150,max_depth=5,learning_rate=.08,n_jobs=4,
                              random_state=SEED,tree_method="hist")
        model.fit(inputs,labels[train])
        del inputs
        model.save_model(directory/f"{name}.json")
        save(name,np.concatenate([model.predict(matrix(chunk),output_margin=True)
                                  for chunk in batches(test,32_768)]))
    for neighbors in (True,False):
        name = "graphsage" if neighbors else "graphsage_no_neighbors"
        model, epoch = fit_sage(features,train,validation,neighbors=neighbors,
                                epochs=epochs,batch_size=batch_size)
        choices[name] = epoch
        torch.save(model.state_dict(),directory/f"{name}.pt")
        save(name,predict(model,features,test,batch_size))
    write_snapshot_index(data.times, data.ids, features.cutoffs[test], directory)
    manifest = dict(runtime={name: version(name) for name in
                             ("numpy", "polars", "torch", "torch-geometric", "scikit-learn", "xgboost")},
                    hardware=platform.machine(), device="cpu", threads=4,
                    wall_seconds=round(perf_counter() - started, 2),
                    run_id=run_id,source_rows=source_rows,scope_end_us=scope_end,source_sha256=fingerprint,feature_version="hourly-v1",seed=SEED,
                    time_basis="synthetic naive epoch microseconds",bucket_us=HOUR,
                    validation_start_us=int(bounds[0]),test_start_us=int(bounds[1]),
                    training_cutoff_us=int(bounds[0]),selection_cutoff_us=int(bounds[1]),
                    graph_policy="Both hops use the same cutoff. Directional means. Repeats contribute multiple messages, including self-edges in both roles.",
                    fanouts_per_direction=FANOUTS,currency_policy="Training vocabulary, per-currency amount history",
                    review_budget=BUDGET,epochs=epochs,selected_epochs=choices,
                    splits={k:dict(rows=int(s.stop-s.start),positives=int(labels[s].sum()),
                                   start_us=int(data.times[s.start]),end_us=int(data.times[s.stop-1]))
                            for k,s in data.splits.items()},metrics=results,
                    artifact_sha256={p.name:hashlib.sha256(p.read_bytes()).hexdigest()
                                     for p in directory.iterdir() if p.suffix in (".pt",".json")})
    (directory/"manifest.json").write_text(json.dumps(manifest,indent=2)+"\n")


def write_snapshot_index(times, ids, cutoffs, directory):
    """Fingerprint the historical source rows behind every scored snapshot."""
    boundaries = np.unique(cutoffs)
    ends = np.searchsorted(times, boundaries, side="left")
    digest = hashlib.sha256()
    previous = 0
    records = []
    for boundary, end in zip(boundaries, ends):
        digest.update(np.asarray(ids[previous:end], dtype="<i8").tobytes())
        previous = end
        records.append({
            "snapshot_id": int(boundary), "context_cutoff": int(boundary),
            "history_start": int(times[0]), "history_rows": int(end),
            "bucket_us": HOUR, "feature_version": "hourly-v1",
            "graph_row_id_sha256": digest.hexdigest(),
            "currency_policy": "training vocabulary; per-currency summaries",
        })
    pl.DataFrame(records).write_parquet(Path(directory) / "snapshot_index.parquet")

In [9]:
SOURCE_SHA256 = "b19d39f515523373f991b689c07e11e7b0b95c17a2c27a87d91584ae16c5b040"

if RUN_FULL_BENCHMARK:
    if not CSV_PATH.is_file():
        raise FileNotFoundError("Set CSV_PATH to the HI-Small transaction CSV.")
    with CSV_PATH.open("rb") as source:
        digest = hashlib.sha256()
        for chunk in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(chunk)
    if digest.hexdigest() != SOURCE_SHA256:
        raise ValueError("The CSV checksum differs from the benchmark checksum.")
    run(CSV_PATH, RUN_DIRECTORY, epochs=EPOCHS, batch_size=BATCH_SIZE)
else:
    print("Training is off. The next section shows results from the saved run.")

Training is off. The next section shows results from the saved run.


## 7. Compare the results

Use the same review budget for each model. Compare average precision across the full ranking. Compare Precision@100 at the review limit.

In [10]:
MODEL_NAMES = {'currency_rule': 'Currency-aware amount rule', 'transaction_xgboost': 'Transaction-only XGBoost', 'graph_xgboost': 'Graph-aware XGBoost', 'graphsage': 'Directed GraphSAGE', 'graphsage_no_neighbors': 'GraphSAGE without neighbors'}
SAVED_METRICS = {'currency_rule': {'average_precision': 0.004294581966052907, 'precision_at_100': 0.0, 'confusion_at_100': {'tp': 0, 'fp': 100, 'fn': 1567, 'tn': 767473}}, 'transaction_xgboost': {'average_precision': 0.13589053309067906, 'precision_at_100': 0.54, 'confusion_at_100': {'tp': 54, 'fp': 46, 'fn': 1513, 'tn': 767527}}, 'graph_xgboost': {'average_precision': 0.4754545209459047, 'precision_at_100': 0.98, 'confusion_at_100': {'tp': 98, 'fp': 2, 'fn': 1469, 'tn': 767571}}, 'graphsage': {'average_precision': 0.24281630445832147, 'precision_at_100': 0.83, 'confusion_at_100': {'tp': 83, 'fp': 17, 'fn': 1484, 'tn': 767556}}, 'graphsage_no_neighbors': {'average_precision': 0.2369522029456851, 'precision_at_100': 0.91, 'confusion_at_100': {'tp': 91, 'fp': 9, 'fn': 1476, 'tn': 767564}}}

if RUN_FULL_BENCHMARK:
    report = json.loads((RUN_DIRECTORY / "manifest.json").read_text())["metrics"]
else:
    report = SAVED_METRICS

results = pl.DataFrame([
    {
        "Model": name,
        "Average precision": round(report[key]["average_precision"], 4),
        "Precision@100": f"{report[key]['precision_at_100']:.0%}",
        "Labeled cases in top 100": report[key]["confusion_at_100"]["tp"],
    }
    for key, name in MODEL_NAMES.items()
])
display(results)

Model,Average precision,Precision@100,Labeled cases in top 100
str,f64,str,i64
"""Currency-aware amount rule""",0.0043,"""0%""",0
"""Transaction-only XGBoost""",0.1359,"""54%""",54
"""Graph-aware XGBoost""",0.4755,"""98%""",98
"""Directed GraphSAGE""",0.2428,"""83%""",83
"""GraphSAGE without neighbors""",0.237,"""91%""",91


### Result

The saved test set contains **769,140 transfers** and **1,567 positive labels**.

Graph-aware XGBoost has the highest AP and Precision@100. Neighbor messages increase GraphSAGE AP from 0.2370 to 0.2428. At 100 alerts, the model with neighbors finds eight fewer labeled cases.

The data is synthetic. The test period overlaps earlier experiments. The hourly cutoff hides activity in the current hour. The scores rank transfers for human review.

### Sources

- [IBM AMLworld HI-Small](https://www.kaggle.com/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml)
- [CDLA-Sharing-1.0](https://cdla.dev/sharing-1-0/)
- [ASD-STE100 Issue 9](https://www.asd-ste100.org/assets/files/ASD-STE100_ISSUE9.pdf)
- OpenAI Codex assisted with code, evaluation, and notebook preparation.